> **Note:** References the old `backend/` layout — kept for reference only.


# End-to-End App Demo
This notebook walks through an end-to-end flow of the app: request  retrieval  ranking  output formatting.

## 1) Setup
Load environment variables and configure paths for local execution.

In [1]:
from pathlib import Path
import os
import json

# Project root (adjust if this notebook is moved)
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
BACKEND_DIR = ROOT / "backend"

print("Root:", ROOT)
print("Data dir:", DATA_DIR)
print("Backend dir:", BACKEND_DIR)

Root: /Users/georgetrump/Visual Studio Code/dishify
Data dir: /Users/georgetrump/Visual Studio Code/dishify/data
Backend dir: /Users/georgetrump/Visual Studio Code/dishify/backend


In [2]:
# Ensure local package imports work when running the notebook
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 2) User request
- Example request payload from client

In [13]:
from backend.app.models.retrieval import ParsedIngredientModel, RetrievalRequest

request_payload = {
    "query": "creamy tomato pasta with spinach",
    "available_ingredients": [
        {
            "name": "penne",
            "quantity": 12,
            "unit": "oz",
            "raw_text": "12 oz penne",
        },
        {
            "name": "tomato",
            "quantity": 4,
            "unit": "count",
            "raw_text": "4 ripe tomatoes",
        },
        {
            "name": "spinach",
            "quantity": 4,
            "unit": "cup",
            "raw_text": "4 cups baby spinach",
        },
        {
            "name": "garlic",
            "quantity": 3,
            "unit": "clove",
            "raw_text": "3 cloves garlic",
        },
        {
            "name": "olive oil",
            "quantity": 2,
            "unit": "tbsp",
            "raw_text": "2 tbsp olive oil",
        },
        {
            "name": "parmesan",
            "quantity": 0.25,
            "unit": "cup",
            "raw_text": "1/4 cup parmesan",
        },
    ],
    "exclusion_restrictions": ["shellfish_allergy", "nut_allergy", "vegetarian"],
}

request = RetrievalRequest(
    query=request_payload["query"],
    available_ingredients=[
        ParsedIngredientModel(**item)
        for item in request_payload["available_ingredients"]
    ],
    exclusion_restrictions=request_payload["exclusion_restrictions"],
)

query = request.query
filters = {
    "exclusion_restrictions": request.exclusion_restrictions,
}

print("Request payload:", request_payload)
print("Request model:", request.dict())
print("Query:", query)
print("Filters:", filters)

Request payload: {'query': 'creamy tomato pasta with spinach', 'available_ingredients': [{'name': 'penne', 'quantity': 12, 'unit': 'oz', 'raw_text': '12 oz penne'}, {'name': 'tomato', 'quantity': 4, 'unit': 'count', 'raw_text': '4 ripe tomatoes'}, {'name': 'spinach', 'quantity': 4, 'unit': 'cup', 'raw_text': '4 cups baby spinach'}, {'name': 'garlic', 'quantity': 3, 'unit': 'clove', 'raw_text': '3 cloves garlic'}, {'name': 'olive oil', 'quantity': 2, 'unit': 'tbsp', 'raw_text': '2 tbsp olive oil'}, {'name': 'parmesan', 'quantity': 0.25, 'unit': 'cup', 'raw_text': '1/4 cup parmesan'}], 'exclusion_restrictions': ['shellfish_allergy', 'peanuts_allergy'], 'dietary_preferences': ['vegetarian']}
Request model: {'query': 'creamy tomato pasta with spinach', 'top_k': 5, 'available_ingredients': [{'name': 'penne', 'quantity': 12.0, 'unit': 'oz', 'raw_text': '12 oz penne'}, {'name': 'tomato', 'quantity': 4.0, 'unit': 'count', 'raw_text': '4 ripe tomatoes'}, {'name': 'spinach', 'quantity': 4.0, 'un

/var/folders/mf/6byhzb_n4lgd1tm_3twbyrlc0000gn/T/ipykernel_40928/2365388336.py:64: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print("Request model:", request.dict())


## 3) Retrieval
Use the vector store to retrieve candidate recipes.


In [ ]:
from backend.app.vector_db.recipe_vector_store import RecipeVectorStore
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "recipes_10000"
top_k = 5

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
client = QdrantClient(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
)

recipe_store = RecipeVectorStore(
    qdrant_client=client,
    embedding_model=model,
    collection_name=COLLECTION_NAME,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5111.19it/s]


#### 3.1) Unfiltered
- Similarity search only uses the query

In [15]:
results = recipe_store.retrieve_recipes(
    query=query,
)

print(f"\nQuery: {query}")
RecipeVectorStore.print_recipes(results, "WITHOUT FILTERS")


Query: creamy tomato pasta with spinach

WITHOUT FILTERS
Retrieved 5 recipes:

#1
Score: 0.70517343
Title: Pasta With Spinach Sauce
Ingredients: 4 slices bacon, cut in 1/2-inch strips, 2 cloves garlic, minced, 6 oz. mushrooms, sliced, 6 scallions, sliced, 1 lb. finely shredded spinach, 2 large tomatoes, peeled and chopped, 1/4 c. minced fresh parsley, salt and pepper, 12 oz. fettuccini noodles, 1 c. whipping cream, Parmesan cheese
Raw ingredients: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
NER: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
Exclusion restrictions: milk_allergy, garlic_allergy, onion_allergy, alpha_gal_syndrome, latex_food_syndrome, lactose_intolerance, histamine_intolerance, fodmap_intolerance, salicylate_sensitivity, tyramine_sensitivity, vegan, ovo_vegetarian, pescatarian, kosher, halal, jain, buddhis

#### 3.2) Filtered: available_ingredients
- Similarity search only uses only the query and available ingredients (without quantities and units to reduce noise)
- We notice a significant improvement in similarity score

In [16]:
results = recipe_store.retrieve_recipes(
    query=query,
    available_ingredients=request.available_ingredients,
)

print(f"\nQuery: {query}")
RecipeVectorStore.print_recipes(
    results,
    "FILTERED: available_ingredients",
)


Query: creamy tomato pasta with spinach

FILTERED: available_ingredients
Retrieved 5 recipes:

#1
Score: 0.8081598
Title: Pasta With Spinach Sauce
Ingredients: 4 slices bacon, cut in 1/2-inch strips, 2 cloves garlic, minced, 6 oz. mushrooms, sliced, 6 scallions, sliced, 1 lb. finely shredded spinach, 2 large tomatoes, peeled and chopped, 1/4 c. minced fresh parsley, salt and pepper, 12 oz. fettuccini noodles, 1 c. whipping cream, Parmesan cheese
Raw ingredients: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
NER: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
Exclusion restrictions: milk_allergy, garlic_allergy, onion_allergy, alpha_gal_syndrome, latex_food_syndrome, lactose_intolerance, histamine_intolerance, fodmap_intolerance, salicylate_sensitivity, tyramine_sensitivity, vegan, ovo_vegetarian, pescatarian, kosher, halal

#### 3.2) Filtered: available_ingredients + exclusion_restrictions
- Similarity search only uses the query and available ingredients (without quantities and units to reduce noise)
- exclusion restrictions are applied as hard filter during the similarity search

In [17]:
results = recipe_store.retrieve_recipes(
    query=query,
    available_ingredients=request.available_ingredients,
    excluded_ingredients=request.exclusion_restrictions,
    top_k=top_k,
)

print(f"\nQuery: {query}")
RecipeVectorStore.print_recipes(
    results,
    "FILTERED: available_ingredients + exclusion_restrictions",
)


Query: creamy tomato pasta with spinach

FILTERED: available_ingredients + exclusion_restrictions
Retrieved 5 recipes:

#1
Score: 0.8081598
Title: Pasta With Spinach Sauce
Ingredients: 4 slices bacon, cut in 1/2-inch strips, 2 cloves garlic, minced, 6 oz. mushrooms, sliced, 6 scallions, sliced, 1 lb. finely shredded spinach, 2 large tomatoes, peeled and chopped, 1/4 c. minced fresh parsley, salt and pepper, 12 oz. fettuccini noodles, 1 c. whipping cream, Parmesan cheese
Raw ingredients: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
NER: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
Exclusion restrictions: milk_allergy, garlic_allergy, onion_allergy, alpha_gal_syndrome, latex_food_syndrome, lactose_intolerance, histamine_intolerance, fodmap_intolerance, salicylate_sensitivity, tyramine_sensitivity, vegan, ovo_vegetarian, p

## 4) Rule-based re-ranking
Re-rank the retrieved recipes using available ingredients.

Final score:
- $\text{final\_score} = 0.70 \cdot \text{semantic\_score} + 0.30 \cdot \text{inventory\_score}$

Examples:
- Full match: recipe needs 0.5 lb sausage, user has 1 lb -> full match.
- Partial match: recipe needs 1 cup milk, user has 0.5 cup -> partial match.
- Missing: recipe needs flour, user has eggs -> missing.

In [18]:
from backend.app.services.ranking import score_recipes_by_inventory

# Re-rank using available ingredients from the request model.
ranked = score_recipes_by_inventory(
    recipes=results,
    available_ingredients=request.available_ingredients,
    ingredient_weight=0.30,
    semantic_weight=0.70,
)

print(f"\nQuery: {query}")
RecipeVectorStore.print_recipes(
    ranked,
    "RANKED with rule-based scoring",
)


Query: creamy tomato pasta with spinach

RANKED with rule-based scoring
Retrieved 5 recipes:

#1
Score: 0.5929845872727272
Title: Pasta With Spinach Sauce
Ingredients: 4 slices bacon, cut in 1/2-inch strips, 2 cloves garlic, minced, 6 oz. mushrooms, sliced, 6 scallions, sliced, 1 lb. finely shredded spinach, 2 large tomatoes, peeled and chopped, 1/4 c. minced fresh parsley, salt and pepper, 12 oz. fettuccini noodles, 1 c. whipping cream, Parmesan cheese
Raw ingredients: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
NER: bacon, garlic, mushrooms, scallions, spinach, tomatoes, fresh parsley, salt, fettuccini noodles, whipping cream, Parmesan cheese
Exclusion restrictions: milk_allergy, garlic_allergy, onion_allergy, alpha_gal_syndrome, latex_food_syndrome, lactose_intolerance, histamine_intolerance, fodmap_intolerance, salicylate_sensitivity, tyramine_sensitivity, vegan, ovo_vegetarian, pescatarian, koshe

## 5) LLM Reasoning & Formatting
Modify and format the top results for UI or API response.

In [20]:
from pathlib import Path
from dotenv import load_dotenv

from backend.app.services.llm_reasoning import generate_reasoning_payload

load_dotenv(Path("../.env"))

# LLM reasoning payload for the top ranked recipes.
reasoning_payload = generate_reasoning_payload(
    request=request,
    recipes=ranked[:top_k],
    provider="openrouter",
)


def _normalize_reasoning_key(value):
    if value is None:
        return None
    return str(value).strip().lower()


results_list = []
if isinstance(reasoning_payload, dict):
    results_list = reasoning_payload.get("results", []) or []
elif isinstance(reasoning_payload, list):
    results_list = reasoning_payload

reasoning_results = {}
for item in results_list:
    if not isinstance(item, dict):
        continue
    item_id = _normalize_reasoning_key(item.get("id"))
    item_title = _normalize_reasoning_key(item.get("title"))
    if item_id:
        reasoning_results[item_id] = item
    if item_title:
        reasoning_results[item_title] = item


def recipe_to_output(recipe, rank):
    key = _normalize_reasoning_key(getattr(recipe, "id", None))
    title_key = _normalize_reasoning_key(getattr(recipe, "title", None))
    reasoning_item = (
        reasoning_results.get(key) or reasoning_results.get(title_key) or {}
    )
    return {
        "rank": rank,
        "id": getattr(recipe, "id", None),
        "title": getattr(recipe, "title", None),
        "summary": getattr(recipe, "summary", None),
        "time_minutes": getattr(recipe, "time_minutes", None),
        "score": getattr(recipe, "score", None),
        "reasoning": reasoning_item.get("reasoning"),
        "directions": getattr(recipe, "directions", None),
    }


output = [recipe_to_output(r, i + 1) for i, r in enumerate(ranked[:top_k])]

print(json.dumps(output, indent=2))

[
  {
    "rank": 1,
    "id": 3136,
    "title": "Pasta With Spinach Sauce",
    "summary": null,
    "time_minutes": null,
    "score": 0.5929845872727272,
    "reasoning": {
      "positive": [],
      "negative": [
        "Contains bacon, which is not vegetarian.",
        "Includes whipping cream and mushrooms that the user may not have.",
        "Uses fettuccini noodles instead of the penne the user has."
      ]
    },
    "directions": [
      "In a heavy pan over moderate heat, cook bacon until fat runs. Add garlic and mushrooms and cook until they begin to lose their juices.",
      "Stir in scallions, spinach, tomatoes and parsley with salt and pepper.",
      "Cover pan and simmer gently for 10 minutes."
    ]
  },
  {
    "rank": 2,
    "id": 6183,
    "title": "Pasta With Spinach, Garlic And Oil",
    "summary": null,
    "time_minutes": null,
    "score": 0.59192187,
    "reasoning": {
      "positive": [
        "All ingredients are vegetarian and contain no shellfish